**Air Quality Prediction: Forecasting the Top 10 Most Polluted Cities in the Next 5 Years**

**Overview**

This project leverages time series forecasting using the ARIMA model to predict air pollution levels across multiple cities. The goal is to identify the top 10 most polluted cities over the next 5 years based on PM2.5 concentrations. The dataset (combined.csv) contains historical pollution data, which is processed and analyzed using Python in a Jupyter Notebook environment.

By accurately forecasting future pollution trends, this project aims to provide valuable insights for policy makers, environmental researchers, and city planners to mitigate pollution and improve air quality.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import warnings
warnings.simplefilter("ignore", ConvergenceWarning)

In [ ]:
### Step 1: Load Dataset
file_path = 'combined.csv'  # Ensure this file is in your working directory
columns = ['datetime', 'cityid', 'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'OZONE']
df = pd.read_csv(file_path, usecols=columns)

**1️⃣ Data Loading & Preprocessing**

Load the dataset (combined.csv) and select relevant columns (datetime, cityid, and pollutant levels like PM2.5, PM10, etc.).

Convert the datetime column to a proper date format for time series analysis.

Handle missing values using mean imputation.

In [ ]:
### Step 2: Convert 'datetime' Column to Date Format
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df.dropna(subset=['datetime'], inplace=True)
df['year_month'] = df['datetime'].dt.to_period('M')

**2️⃣ Data Aggregation**

Convert timestamps to monthly periods (year_month) to perform monthly pollution trend analysis.

Group pollution data by city and month, computing average pollution levels per month for each city


In [ ]:
### Step 3: Handle Missing Values
pollutant_cols = ['PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'OZONE']
df[pollutant_cols] = df[pollutant_cols].fillna(df[pollutant_cols].mean())

**3️⃣ Ensuring Data Variability**

Cities with constant PM2.5 values are skipped, as ARIMA cannot process non-varying data.

Cities with less than 12 months of data are also ignored to ensure robust forecasting.


In [ ]:
### Step 4: Aggregate Pollution Levels per City-Month
city_pollution = df.groupby(['cityid', 'year_month'])[pollutant_cols].mean().reset_index()
city_pollution['year_month'] = city_pollution['year_month'].astype(str)
city_pollution['year_month'] = pd.to_datetime(city_pollution['year_month'])

**4️⃣ Time Series Forecasting with ARIMA**

The ARIMA (AutoRegressive Integrated Moving Average) model is applied to forecast pollution levels.

The model predicts pollution trends for each city over the next 5 years.

Error handling ensures that unstable models are skipped, preventing failures


In [ ]:
### Step 5: Visualize AQI Trends
plt.figure(figsize=(12,6))
for city in city_pollution['cityid'].unique()[:5]:  # Plotting first 5 cities
    city_data = city_pollution[city_pollution['cityid'] == city]
    plt.plot(city_data['year_month'], city_data['PM2.5'], label=f'City {city}')
plt.xlabel('Year-Month')
plt.ylabel('PM2.5 Levels')
plt.title('PM2.5 Trends Over Time')
plt.legend()
plt.show()

In [ ]:
### Step 6: Forecast Pollution using ARIMA
def forecast_pollution(city_data, pollutant, forecast_steps=60):
    city_data.set_index('year_month', inplace=True)
    model = ARIMA(city_data[pollutant], order=(5, 1, 0))
    model_fit = model.fit()
    forecast = model_fit.forecast(steps=forecast_steps)
    return forecast

In [ ]:
### Step 7: Predict Top 10 Most Polluted Cities
def predict_top_cities(df):
    future_pollution = {}
    pollutant = 'PM2.5'  # Key indicator
    cities = df['cityid'].unique()

    for city in cities:
        city_data = df[df['cityid'] == city].copy()
        print(f"Processing City: {city}, Data Points: {len(city_data)}")  # Debugging output

        if len(city_data) < 12:  # Allow more cities by lowering the threshold
            print(f"Skipping City {city}, insufficient data")
            continue

        forecasted_values = forecast_pollution(city_data, pollutant)
        print(f"Forecasted PM2.5 for City {city}: {forecasted_values[:5]}")  # Debugging output

        avg_future_pollution = np.mean(forecasted_values)
        future_pollution[city] = avg_future_pollution

    top_cities = sorted(future_pollution.items(), key=lambda x: x[1], reverse=True)[:10]
    return top_cities

**5️⃣ Extracting the Top 10 Most Polluted Cities**

Forecasted pollution levels are averaged over the predicted period.

In [ ]:
### Step 8: Run Analysis
top_cities = predict_top_cities(city_pollution)
print("\nTop 10 most polluted cities for the next 5 years:")
for city, pollution in top_cities:
    print(f"{city}: {pollution}")

The top 10 cities with the highest projected PM2.5 levels are identified and displayed.
